# 01. Data Collection and Cleaning

Download daily OHLCV from Yahoo Finance for ~750 instruments across 11
sectors, clean the panel, and persist as Parquet.

**Output:** `cleaned_data.parquet` (~30 MB, 5× smaller than the previous
pickle artifact).

In [ ]:
import logging

import pandas as pd
import plotly.io as pio
import yfinance as yf

import quant_sector_optimizer as qso
from quant_sector_optimizer import plotting

pio.renderers.default = "png"
logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

## 1. Download

`yf.Sector` exposes the top ETFs, mutual funds and companies per sector.
We collect their tickers, then run a single batched `yf.download` for speed.

In [ ]:
START, END = "2010-01-01", "2024-12-31"
SECTORS = [
    "basic-materials", "communication-services", "consumer-cyclical",
    "consumer-defensive", "energy", "financial-services", "healthcare",
    "industrials", "real-estate", "technology", "utilities",
]

ticker_metadata, all_tickers = [], set()
for sector in SECTORS:
    obj = yf.Sector(sector)
    instruments = {
        "etfs": obj.top_etfs,
        "mutual_funds": obj.top_mutual_funds,
        "companies": obj.top_companies,
    }
    for category, inst in instruments.items():
        if inst is None or len(inst) == 0:
            continue
        tickers = list(inst.keys()) if category != "companies" else list(inst.index)
        for t in tickers:
            all_tickers.add(t)
            ticker_metadata.append({"Ticker": t, "Sector": sector, "Category": category})

print(f"{len(all_tickers)} unique tickers across {len(SECTORS)} sectors")

In [ ]:
batch = yf.download(
    list(all_tickers), start=START, end=END,
    group_by="ticker", auto_adjust=True, progress=True, threads=True,
)

frames, failed = [], []
for ticker in batch.columns.levels[0]:
    sub = batch[ticker].copy()
    if sub.dropna(how="all").empty:
        failed.append(ticker)
        continue
    sub["Ticker"] = ticker
    sub.reset_index(inplace=True)
    frames.append(sub)

df = pd.concat(frames, ignore_index=True).dropna(subset=["Close"])
meta = pd.DataFrame(ticker_metadata).drop_duplicates(subset=["Ticker"])
df = df.merge(meta, on="Ticker", how="left")
print(f"{df.Ticker.nunique()} tickers loaded ({len(failed)} failed)")

## 2. Clean

`qso.clean_ohlc` drops non-positive prices, drops absurd outliers, and
**reports** the High/Low corrections it applies (no longer silent).

In [ ]:
df, audit = qso.clean_ohlc(df)
print("Cleaning audit:", audit)

# Documented manual correction: TFIFX missing 2016-12-05.
df = qso.fix_missing_date(df, ticker="TFIFX", missing_date="2016-12-05", copy_from_date="2016-12-06")

## 3. Coverage analysis

For each ticker we record first/last date and the share of expected business
days that are missing.

In [ ]:
expected_dates = pd.bdate_range(START, END)
coverage_rows = []
for ticker, g in df.groupby("Ticker", observed=True):
    available = pd.DatetimeIndex(g["Date"]).normalize()
    span_dates = pd.bdate_range(available.min(), available.max())
    missing = span_dates.difference(available)
    coverage_rows.append({
        "Ticker": ticker,
        "Sector": g["Sector"].iloc[0],
        "Category": g["Category"].iloc[0],
        "first_date": available.min(),
        "last_date": available.max(),
        "duration_years": (available.max() - available.min()).days / 365.25,
        "missing_pct_in_period": 100.0 * len(missing) / max(len(span_dates), 1),
        "missing_pct_overall": 100.0 * (len(expected_dates) - len(available)) / len(expected_dates),
    })
coverage = pd.DataFrame(coverage_rows)
coverage["complete_coverage"] = coverage["missing_pct_overall"] < 1.0
coverage.head()

In [ ]:
plotting.plot_universe_growth(coverage, mode="absolute").show()
plotting.plot_coverage_summary(coverage).show()
plotting.plot_sector_category_distribution(df).show()

## 4. Drop short-lived tickers and persist

In [ ]:
short_tickers = coverage.loc[coverage["duration_years"] < 1.0, "Ticker"].tolist()
df = df[~df["Ticker"].isin(short_tickers)].copy()
print(f"Dropped {len(short_tickers)} tickers with < 1 year of history")
print(f"Final: {df['Ticker'].nunique()} tickers, {len(df):,} observations, "
      f"{df['Date'].min().date()} → {df['Date'].max().date()}")

qso.save_panel(df, "cleaned_data.parquet")
print("Saved cleaned_data.parquet")